# Fine-tune Student with HE-friendly Approximations

Light recovery fine-tuning of the distilled student **after** replacing LayerNorm and other nonlinearities with HE-friendly approximations (ApproxLayerNorm, polynomial GELU, HE attention).

- Teacher stays frozen and exact (no approximations).
- Student uses **ApproxLayerNorm** (scaled-Newton inverse sqrt, iters=6) and existing HE-friendly nonlinearities **during training**.
- Distillation loss and splits match the original training setup so results are comparable.


In [ ]:
# Colab setup: clone repo, install deps, device
import sys
from pathlib import Path

REPO_URL = "https://github.com/PulockDas/Secure-Inference-Token-Reduced-VIT.git"
PROJECT_DIR = "/content/Secure-Inference-Token-Reduced-VIT"

if not Path(PROJECT_DIR).exists():
    !git clone $REPO_URL $PROJECT_DIR
%cd $PROJECT_DIR
!git fetch origin
!git checkout feature/he-inference
!git reset --hard origin/feature/he-inference
!pip install -q -r $PROJECT_DIR/requirements.txt

sys.path.insert(0, PROJECT_DIR)
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
# Mount Google Drive and define paths
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE_DRIVE_DIR = "/content/drive/MyDrive/Secure-Inference-Token-Reduced-VIT"

# Pretrained distilled student (original training with LayerNorm)
STUDENT_CKPT_IN = f"{BASE_DRIVE_DIR}/checkpoints/student_best_K97_E60.pt"

# Teacher checkpoint
TEACHER_CKPT = f"{BASE_DRIVE_DIR}/checkpoints/teacher_best.pt"

# Output fine-tuned approx student
RUN_NAME = "student_approx_finetuned_K97_E5"
STUDENT_CKPT_OUT = f"{BASE_DRIVE_DIR}/checkpoints/{RUN_NAME}.pt"

# Logs and results
LOG_CSV = f"{BASE_DRIVE_DIR}/logs/{RUN_NAME}.csv"
RESULTS_DIR = f"{BASE_DRIVE_DIR}/results/{RUN_NAME}"

for p in [
    Path(BASE_DRIVE_DIR) / "checkpoints",
    Path(BASE_DRIVE_DIR) / "logs",
    Path(BASE_DRIVE_DIR) / "results",
]:
    p.mkdir(parents=True, exist_ok=True)

print("Student in:", STUDENT_CKPT_IN)
print("Teacher ckpt:", TEACHER_CKPT)
print("Student out:", STUDENT_CKPT_OUT)


In [ ]:
# Reload project modules (pick up latest ApproxLayerNorm, training code)
import sys

for k in list(sys.modules.keys()):
    if k.startswith("models") or k.startswith("training") or k.startswith("data"):
        del sys.modules[k]

!find $PROJECT_DIR -name __pycache__ -type d -exec rm -rf {} + 2>/dev/null; echo "cache cleared"

from data import get_lc25000_root, get_dataloaders
from training import load_student_checkpoint, load_teacher_checkpoint, distillation_loss, evaluate_teacher
from models import replace_layernorm_with_approx
import torch.nn as nn


In [ ]:
# Data: LC25000 train/val/test (same split as original training)
root = get_lc25000_root()
train_loader, val_loader, test_loader = get_dataloaders(
    root_dir=root,
    batch_size=32,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
    subdir_depth=2,
    image_size=224,
    num_workers=2,
)

ds = train_loader.dataset
num_classes = ds.num_classes
print("Classes:", ds.class_names)
print("Train:", len(ds), "Val:", len(val_loader.dataset), "Test:", len(test_loader.dataset))


In [ ]:
# Load teacher (frozen, exact) and student (then replace LN with ApproxLayerNorm)
teacher = load_teacher_checkpoint(TEACHER_CKPT, device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
print("Teacher loaded.")

student = load_student_checkpoint(STUDENT_CKPT_IN, device, norm_mode="layernorm")
student.eval()
print("Student (LayerNorm) loaded.")

# Calibration subset for ApproxLayerNorm (one-time, no recalibration during training)
from torch.utils.data import Subset, DataLoader

NUM_CALIBRATION_BATCHES = 64
calib_size = min(NUM_CALIBRATION_BATCHES * train_loader.batch_size, len(train_loader.dataset))
calib_indices = list(range(calib_size))
calib_dataset = Subset(train_loader.dataset, calib_indices)
calib_loader = DataLoader(
    calib_dataset,
    batch_size=train_loader.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)
print("Calibration samples:", len(calib_dataset))

replaced = replace_layernorm_with_approx(
    student,
    calib_loader,
    device,
    num_calibration_batches=NUM_CALIBRATION_BATCHES,
    eps=1e-6,
    iters=6,
)
print("Replaced LayerNorm modules with ApproxLayerNorm:", replaced)
student.train()


In [ ]:
# Fine-tuning loop: call shared train_student_with_approx from training module
import csv
from pathlib import Path
from training import train_student_with_approx

result = train_student_with_approx(
    student=student,
    teacher=teacher,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=5,
    lr=5e-6,
    temperature=4.0,
    alpha=0.7,
)
student = result["student"]
history = result["history"]
best_val_acc = result["best_val_acc"]
print("Best val acc:", best_val_acc)


In [ ]:
# Save fine-tuned student and logs; evaluate on test set
import json
from pathlib import Path

ckpt_dir = Path(BASE_DRIVE_DIR) / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

state = student.state_dict()
ckpt = {
    "student_state_dict": state,
    "epoch": history[-1]["epoch"],
    "val_acc": best_val_acc,
    "num_classes": ds.num_classes,
    "num_output_tokens": getattr(student.token_reduction, "num_output_tokens", 97),
    "embed_dim": student.embed_dim,
    "depth": len(student.blocks),
    "num_heads": student.blocks[0].attn.num_heads,
    "norm_mode": "layernorm",
}
torch.save(ckpt, STUDENT_CKPT_OUT)
print("Saved fine-tuned approx student to:", STUDENT_CKPT_OUT)

# Save compact training log
log_path = Path(LOG_CSV)
log_path.parent.mkdir(parents=True, exist_ok=True)
with open(log_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "train_acc", "val_acc"])
    writer.writeheader()
    for row in history:
        writer.writerow({
            "epoch": row["epoch"],
            "train_loss": row["train_loss"],
            "train_acc": row["train_acc"],
            "val_acc": row["val_acc"],
        })
print("Saved training log to:", LOG_CSV)

# Final evaluation on test set (saves JSON + txt summary under RESULTS_DIR)
test_result = evaluate_teacher(
    student,
    test_loader,
    device,
    ds.class_names,
    results_dir=RESULTS_DIR,
    results_subdir="student_approx_finetuned",
)
print("Test accuracy:", f"{test_result['test_acc']:.4f}")
print("Per-class test accuracy:", [f"{a:.4f}" for a in test_result["per_class_acc"]])


In [ ]:
# Save fine-tuned checkpoint to Drive and push logs/results to GitHub (no checkpoints)
import shutil
from pathlib import Path

# 1) Copy best fine-tuned student checkpoint to Drive only
DRIVE_CKPT_DIR = f"{BASE_DRIVE_DIR}/checkpoints"
Path(DRIVE_CKPT_DIR).mkdir(parents=True, exist_ok=True)

src_ckpt = STUDENT_CKPT_OUT
if Path(src_ckpt).exists():
    dst_ckpt = Path(DRIVE_CKPT_DIR) / Path(src_ckpt).name
    shutil.copy2(src_ckpt, dst_ckpt)
    print("Fine-tuned student checkpoint saved to Drive:", dst_ckpt)
else:
    print("WARNING: Expected checkpoint not found at", src_ckpt)

# 2) Push only logs and results (no checkpoints) to GitHub
GITHUB_TOKEN = "your_token_here"  # <- set this before pushing
%cd $PROJECT_DIR
!git config --global user.name "PulockDas"
!git config --global user.email "pulockkamol50@gmail.com"
!git add results/ logs/
!git commit -m "Add approx-student fine-tuning logs and evaluation" || echo "No changes to commit"
# Uncomment the line below after setting GITHUB_TOKEN
# !git push https://{GITHUB_TOKEN}@github.com/PulockDas/Secure-Inference-Token-Reduced-VIT.git
